# 📊 Data Preprocessing - Political News Sentiment Analysis

**Thesis:** Analysis of the Impact of Political News on LQ45 Stock Price Index Volatility  
**Period:** 2019-2024  
**Sources:** CNBC Indonesia + Kompas  

**Preprocessing Steps:**
1. Load & merge datasets
2. Date filtering (2019-2024 only!)
3. Text cleaning
4. Stopword removal
5. Normalization (Sastrawi stemming)
6. Remove duplicates
7. Final dataset ready for IndoBERT!

---

## 📦 Step 1: Install & Import Libraries

In [1]:
# Install required packages
!pip install pandas numpy sastrawi nltk tqdm --quiet

print("✅ Packages installed!")

✅ Packages installed!



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import re
from datetime import datetime
from tqdm import tqdm

# Sastrawi for Indonesian stemming
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# NLTK
import nltk
nltk.download('punkt', quiet=True)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

print("✅ Libraries imported!")

✅ Libraries imported!


## 📁 Step 2: Load Datasets

In [3]:
print("="*70)
print("📥 LOADING DATASETS")
print("="*70)

# Load CNBC
print("\n1. Loading CNBC Indonesia...")
df_cnbc = pd.read_csv('CNBC_MERGED_2019-2024.csv', encoding='utf-8-sig')
print(f"   ✅ CNBC: {len(df_cnbc):,} articles")
print(f"   Columns: {df_cnbc.columns.tolist()}")

# Load Kompas
print("\n2. Loading Kompas...")
df_kompas = pd.read_csv('kompas_politik_indonesia_articles.csv')
print(f"   ✅ Kompas: {len(df_kompas):,} articles")
print(f"   Columns: {df_kompas.columns.tolist()}")

print(f"\n📊 Total raw data: {len(df_cnbc) + len(df_kompas):,} articles")

📥 LOADING DATASETS

1. Loading CNBC Indonesia...
   ✅ CNBC: 5,343 articles
   Columns: ['url', 'title', 'content', 'content_length', 'scraped_at', 'date']

2. Loading Kompas...
   ✅ Kompas: 9,980 articles
   Columns: ['title', 'tanggal', 'content']

📊 Total raw data: 15,323 articles


## 🔄 Step 3: Standardize Column Names & Structure

In [5]:
print("="*70)
print("🔄 STANDARDIZING STRUCTURE")
print("="*70)

# CNBC: Rename columns
df_cnbc_clean = df_cnbc[[ 'title', 'content', 'date']].copy()
df_cnbc_clean['source'] = 'CNBC'

# Kompas: Rename columns
df_kompas_clean = df_kompas.rename(columns={'tanggal': 'date'}).copy()
df_kompas_clean = df_kompas_clean[['title', 'content', 'date']]
df_kompas_clean['source'] = 'Kompas'

print("✅ Columns standardized!")
print(f"\n   Standard columns: ['title', 'content', 'date', 'source']")

🔄 STANDARDIZING STRUCTURE
✅ Columns standardized!

   Standard columns: ['title', 'content', 'date', 'source']


    ## 📅 Step 4: Parse & Filter Dates (2019-2024 ONLY!)

In [6]:
print("="*70)
print("📅 DATE PARSING & FILTERING")
print("="*70)

def parse_date_cnbc(date_str):
    """Parse CNBC dates (YYYY-MM-DD)"""
    try:
        return pd.to_datetime(date_str, format='%Y-%m-%d')
    except:
        return None

def parse_date_kompas(date_str):
    """Parse Kompas dates (DD Month YYYY in Indonesian)"""
    try:
        # Map Indonesian months to numbers
        months = {
            'Januari': '01', 'Februari': '02', 'Maret': '03', 'April': '04',
            'Mei': '05', 'Juni': '06', 'Juli': '07', 'Agustus': '08',
            'September': '09', 'Oktober': '10', 'November': '11', 'Desember': '12'
        }
        
        parts = str(date_str).split()
        if len(parts) == 3:
            day, month_name, year = parts
            month = months.get(month_name)
            if month:
                date_str_eng = f"{year}-{month}-{day.zfill(2)}"
                return pd.to_datetime(date_str_eng, format='%Y-%m-%d')
    except:
        pass
    return None

# Parse dates
print("\n1. Parsing CNBC dates...")
df_cnbc_clean['date'] = df_cnbc_clean['date'].apply(parse_date_cnbc)
before = len(df_cnbc_clean)
df_cnbc_clean = df_cnbc_clean.dropna(subset=['date'])
print(f"   Parsed: {len(df_cnbc_clean)}/{before} (removed {before - len(df_cnbc_clean)} invalid)")

print("\n2. Parsing Kompas dates...")
df_kompas_clean['date'] = df_kompas_clean['date'].apply(parse_date_kompas)
before = len(df_kompas_clean)
df_kompas_clean = df_kompas_clean.dropna(subset=['date'])
print(f"   Parsed: {len(df_kompas_clean)}/{before} (removed {before - len(df_kompas_clean)} invalid)")

# Filter 2019-2024 ONLY
print("\n3. Filtering dates: 2019-09-01 to 2024-09-30")
start_date = pd.to_datetime('2019-09-01')
end_date = pd.to_datetime('2024-09-30')

before_cnbc = len(df_cnbc_clean)
df_cnbc_clean = df_cnbc_clean[(df_cnbc_clean['date'] >= start_date) & (df_cnbc_clean['date'] <= end_date)]
print(f"   CNBC: {len(df_cnbc_clean)}/{before_cnbc} (removed {before_cnbc - len(df_cnbc_clean)} outside range)")

before_kompas = len(df_kompas_clean)
df_kompas_clean = df_kompas_clean[(df_kompas_clean['date'] >= start_date) & (df_kompas_clean['date'] <= end_date)]
print(f"   Kompas: {len(df_kompas_clean)}/{before_kompas} (removed {before_kompas - len(df_kompas_clean)} outside range)")

# Show date ranges
print(f"\n📊 Date ranges after filtering:")
print(f"   CNBC: {df_cnbc_clean['date'].min().date()} to {df_cnbc_clean['date'].max().date()}")
print(f"   Kompas: {df_kompas_clean['date'].min().date()} to {df_kompas_clean['date'].max().date()}")

📅 DATE PARSING & FILTERING

1. Parsing CNBC dates...
   Parsed: 5343/5343 (removed 0 invalid)

2. Parsing Kompas dates...
   Parsed: 9980/9980 (removed 0 invalid)

3. Filtering dates: 2019-09-01 to 2024-09-30
   CNBC: 5343/5343 (removed 0 outside range)
   Kompas: 7155/9980 (removed 2825 outside range)

📊 Date ranges after filtering:
   CNBC: 2019-09-01 to 2024-09-30
   Kompas: 2022-01-20 to 2024-09-30


## 🔗 Step 5: Merge Datasets

In [7]:
print("="*70)
print("🔗 MERGING DATASETS")
print("="*70)

# Combine
df_merged = pd.concat([df_cnbc_clean, df_kompas_clean], ignore_index=True)

print(f"\n✅ Merged dataset: {len(df_merged):,} articles")
print(f"\n📊 Source distribution:")
print(df_merged['source'].value_counts())

print(f"\n📅 Date range: {df_merged['date'].min().date()} to {df_merged['date'].max().date()}")

🔗 MERGING DATASETS

✅ Merged dataset: 12,498 articles

📊 Source distribution:
source
Kompas    7155
CNBC      5343
Name: count, dtype: int64

📅 Date range: 2019-09-01 to 2024-09-30


## 🧹 Step 6: Text Cleaning

In [8]:
print("="*70)
print("🧹 TEXT CLEANING")
print("="*70)

def clean_text(text):
    """
    Clean Indonesian text:
    1. Convert to lowercase
    2. Remove URLs
    3. Remove emails
    4. Remove special characters (keep Indonesian letters)
    5. Remove extra whitespace
    6. Remove numbers
    """
    if pd.isna(text) or text == "N/A":
        return ""
    
    text = str(text)
    
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # Remove emails
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove newlines and tabs
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    
    # Remove special characters (keep spaces and Indonesian letters)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

# Apply cleaning
print("\nCleaning text (this may take a few minutes)...")
tqdm.pandas(desc="Cleaning")
df_merged['text_clean'] = df_merged['content'].progress_apply(clean_text)

# Remove empty content
before = len(df_merged)
df_merged = df_merged[df_merged['text_clean'].str.len() > 50]  # Minimum 50 chars
print(f"\n✅ Cleaned! Removed {before - len(df_merged)} articles with insufficient content")
print(f"   Remaining: {len(df_merged):,} articles")

# Show sample
print(f"\n📋 Sample before/after:")
sample = df_merged.iloc[0]
print(f"\nOriginal (first 200 chars):")
print(sample['content'][:200])
print(f"\nCleaned (first 200 chars):")
print(sample['text_clean'][:200])

🧹 TEXT CLEANING

Cleaning text (this may take a few minutes)...


Cleaning: 100%|██████████| 12498/12498 [00:02<00:00, 5696.57it/s]


✅ Cleaned! Removed 0 articles with insufficient content
   Remaining: 12,498 articles

📋 Sample before/after:

Original (first 200 chars):
Jakarta, CNBC Indonesia-Pemerintah terus mendorong peningkatan investasi melalui penerapan Omnibus Law dengan memangkas dan menyederhanakan regulasi terkait izin usaha dan investasi yang dilaksanakan 

Cleaned (first 200 chars):
jakarta cnbc indonesiapemerintah terus mendorong peningkatan investasi melalui penerapan omnibus law dengan memangkas dan menyederhanakan regulasi terkait izin usaha dan investasi yang dilaksanakan un


## 🛑 Step 7: Stopword Removal

In [9]:
print("="*70)
print("🛑 STOPWORD REMOVAL")
print("="*70)

# Initialize Sastrawi stopword remover
stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

# Get stopword list
stopwords = stopword_factory.get_stop_words()
print(f"\nUsing {len(stopwords)} Indonesian stopwords")
print(f"Sample stopwords: {list(stopwords)[:20]}")

# Apply stopword removal
print("\nRemoving stopwords (this may take a while)...")
tqdm.pandas(desc="Stopword removal")
df_merged['text_no_stopwords'] = df_merged['text_clean'].progress_apply(stopword_remover.remove)

print("\n✅ Stopwords removed!")

# Show sample
print(f"\n📋 Sample before/after stopword removal:")
sample = df_merged.iloc[0]
print(f"\nBefore (first 150 chars):")
print(sample['text_clean'][:150])
print(f"\nAfter (first 150 chars):")
print(sample['text_no_stopwords'][:150])

🛑 STOPWORD REMOVAL

Using 126 Indonesian stopwords
Sample stopwords: ['yang', 'untuk', 'pada', 'ke', 'para', 'namun', 'menurut', 'antara', 'dia', 'dua', 'ia', 'seperti', 'jika', 'jika', 'sehingga', 'kembali', 'dan', 'tidak', 'ini', 'karena']

Removing stopwords (this may take a while)...


Stopword removal: 100%|██████████| 12498/12498 [00:06<00:00, 2028.63it/s]


✅ Stopwords removed!

📋 Sample before/after stopword removal:

Before (first 150 chars):
jakarta cnbc indonesiapemerintah terus mendorong peningkatan investasi melalui penerapan omnibus law dengan memangkas dan menyederhanakan regulasi ter

After (first 150 chars):
jakarta cnbc indonesiapemerintah terus mendorong peningkatan investasi melalui penerapan omnibus law memangkas menyederhanakan regulasi terkait izin u


## 🌱 Step 8: Stemming (Normalization)

In [10]:
print("="*70)
print("🌱 STEMMING (SASTRAWI)")
print("="*70)

# Initialize Sastrawi stemmer
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

print("\nUsing Sastrawi (Nazief & Adriani algorithm)")
print("This process may take 10-15 minutes for large datasets...\n")

# Apply stemming
tqdm.pandas(desc="Stemming")
df_merged['text_stemmed'] = df_merged['text_no_stopwords'].progress_apply(stemmer.stem)

# Remove if too short after stemming
before = len(df_merged)
df_merged = df_merged[df_merged['text_stemmed'].str.len() > 30]
print(f"\n✅ Stemming complete! Removed {before - len(df_merged)} articles too short after stemming")
print(f"   Remaining: {len(df_merged):,} articles")

# Show sample
print(f"\n📋 Sample before/after stemming:")
sample = df_merged.iloc[0]
print(f"\nBefore stemming (first 150 chars):")
print(sample['text_no_stopwords'][:150])
print(f"\nAfter stemming (first 150 chars):")
print(sample['text_stemmed'][:150])

🌱 STEMMING (SASTRAWI)

Using Sastrawi (Nazief & Adriani algorithm)
This process may take 10-15 minutes for large datasets...



Stemming:   2%|▏         | 206/12498 [07:56<7:53:33,  2.31s/it] 


KeyboardInterrupt: 

## 🔄 Step 9: Remove Duplicates

In [ ]:
print("="*70)
print("🔄 REMOVING DUPLICATES")
print("="*70)

before = len(df_merged)

# Method 1: Remove exact title duplicates
df_merged = df_merged.drop_duplicates(subset=['title'], keep='first')
after_title = len(df_merged)
print(f"\n1. By title: Removed {before - after_title} duplicates")

# Method 2: Remove very similar content (first 100 chars)
df_merged['content_hash'] = df_merged['text_clean'].str[:100]
df_merged = df_merged.drop_duplicates(subset=['content_hash'], keep='first')
after_content = len(df_merged)
print(f"2. By content: Removed {after_title - after_content} duplicates")

df_merged = df_merged.drop(columns=['content_hash'])

print(f"\n✅ Total removed: {before - after_content} duplicates")
print(f"   Final dataset: {len(df_merged):,} unique articles")

## 📊 Step 10: Final Statistics & Validation

In [ ]:
print("="*70)
print("📊 FINAL DATASET STATISTICS")
print("="*70)

# Basic stats
print(f"\n✅ Total articles: {len(df_merged):,}")
print(f"\n📅 Date range:")
print(f"   From: {df_merged['date'].min().date()}")
print(f"   To: {df_merged['date'].max().date()}")
print(f"   Days: {(df_merged['date'].max() - df_merged['date'].min()).days}")

print(f"\n📰 Source distribution:")
source_dist = df_merged['source'].value_counts()
for source, count in source_dist.items():
    print(f"   {source}: {count:,} ({count/len(df_merged)*100:.1f}%)")

print(f"\n📈 Monthly distribution (sample):")
df_merged['year_month'] = df_merged['date'].dt.to_period('M')
monthly = df_merged['year_month'].value_counts().sort_index()
print(f"\n   First 6 months:")
for period, count in monthly.head(6).items():
    print(f"   {period}: {count:4d} articles")
print(f"\n   Last 6 months:")
for period, count in monthly.tail(6).items():
    print(f"   {period}: {count:4d} articles")

print(f"\n📏 Text length statistics:")
df_merged['text_length'] = df_merged['text_stemmed'].str.len()
print(f"   Mean: {df_merged['text_length'].mean():.0f} chars")
print(f"   Median: {df_merged['text_length'].median():.0f} chars")
print(f"   Min: {df_merged['text_length'].min():.0f} chars")
print(f"   Max: {df_merged['text_length'].max():.0f} chars")

## 💾 Step 11: Save Preprocessed Data

In [ ]:
print("="*70)
print("💾 SAVING PREPROCESSED DATA")
print("="*70)

# Select final columns
df_final = df_merged[[
    'date',
    'title',
    'source',
    'content',  # Original
    'text_clean',  # Cleaned
    'text_stemmed',  # Ready for IndoBERT!
    'text_length'
]].copy()

# Sort by date
df_final = df_final.sort_values('date').reset_index(drop=True)

# Save
output_file = 'political_news_preprocessed_2019_2024.csv'
df_final.to_csv(output_file, index=False, encoding='utf-8-sig')

import os
file_size = os.path.getsize(output_file) / (1024**2)

print(f"\n✅ Saved: {output_file}")
print(f"   Rows: {len(df_final):,}")
print(f"   Columns: {len(df_final.columns)}")
print(f"   Size: {file_size:.2f} MB")

print(f"\n📋 Column descriptions:")
print(f"   - date: Article publication date")
print(f"   - title: Article title")
print(f"   - source: CNBC or Kompas")
print(f"   - content: Original text")
print(f"   - text_clean: Cleaned text")
print(f"   - text_stemmed: Ready for IndoBERT! ⭐")
print(f"   - text_length: Character count")

print(f"\n📊 Sample data:\n")
display(df_final[['date', 'title', 'text_length', 'source']].head(10))

## 📈 Step 12: Data Quality Report

In [ ]:
print("="*70)
print("📈 DATA QUALITY REPORT")
print("="*70)

# Missing values
print(f"\n1. Missing Values:")
missing = df_final.isnull().sum()
if missing.sum() == 0:
    print(f"   ✅ No missing values!")
else:
    print(missing[missing > 0])

# Date coverage
print(f"\n2. Date Coverage:")
date_range = pd.date_range(start='2019-09-01', end='2024-09-30', freq='D')
actual_dates = df_final['date'].dt.date.unique()
coverage = len(actual_dates) / len(date_range) * 100
print(f"   Days with articles: {len(actual_dates):,} / {len(date_range):,}")
print(f"   Coverage: {coverage:.1f}%")

# Text quality
print(f"\n3. Text Quality:")
very_short = len(df_final[df_final['text_length'] < 100])
short = len(df_final[(df_final['text_length'] >= 100) & (df_final['text_length'] < 500)])
medium = len(df_final[(df_final['text_length'] >= 500) & (df_final['text_length'] < 1000)])
long = len(df_final[df_final['text_length'] >= 1000])

print(f"   Very short (<100): {very_short:,} ({very_short/len(df_final)*100:.1f}%)")
print(f"   Short (100-500): {short:,} ({short/len(df_final)*100:.1f}%)")
print(f"   Medium (500-1000): {medium:,} ({medium/len(df_final)*100:.1f}%)")
print(f"   Long (>1000): {long:,} ({long/len(df_final)*100:.1f}%)")

# Distribution balance
print(f"\n4. Temporal Distribution:")
yearly = df_final.groupby(df_final['date'].dt.year).size()
print(f"   Articles per year:")
for year, count in yearly.items():
    print(f"   {year}: {count:,} articles")

# Final verdict
print(f"\n" + "="*70)
print("✅ DATA QUALITY: GOOD")
print("="*70)
print(f"\n📌 Dataset ready for IndoBERT sentiment analysis!")
print(f"📌 Use column 'text_stemmed' for model input")
print(f"📌 Total: {len(df_final):,} preprocessed articles")
print(f"📌 Period: 2019-09-01 to 2024-09-30")

---

## ✅ PREPROCESSING COMPLETE!

### Next Steps:
1. **Load preprocessed data:** `political_news_preprocessed_2019_2024.csv`
2. **Sentiment Analysis:** Use IndoBERT on `text_stemmed` column
3. **Merge with LQ45:** Join by date
4. **Train BiLSTM:** Predict volatility!

### Files Created:
- `political_news_preprocessed_2019_2024.csv` ⭐ **MAIN OUTPUT**

---